# 2.6 — Clustering (KMeans) et réduction de dimension (ACP) — version Rust

**Navigation** : [<< 2.5-Biais-Variance-CV-ROC](2.5-Biais-Variance-CV-ROC.ipynb) | Version Python : [2.6-Clustering-KMeans-PCA](2.6-Clustering-KMeans-PCA.ipynb) | [2.7-Modeles-Non-Parametriques >>](2.7-Modeles-Non-Parametriques.ipynb) | [Index](../README.md)

**Kernel** : Rust (evcxr)

> **Jumeau Rust.** Ce carnet est le portage du carnet Python [2.6-Clustering-KMeans-PCA](2.6-Clustering-KMeans-PCA.ipynb) : même fil pédagogique, même jeu de données (les 1797 chiffres manuscrits de `load_digits`, lus ici dans `smartcore::dataset::digits`), mêmes valeurs de `k`, de `eps` et de nombre de composantes. Cinq différences de forme, à connaître avant de lire :
>
> - la bibliothèque est **`smartcore` 0.7.0** à la place de scikit-learn ; les générateurs de la section 8 (`make_moons`, `make_circles`) y sont réécrits à la main, avec le générateur déjà utilisé par le jumeau [2.7](2.7-Modeles-Non-Parametriques-Rust.ipynb) ;
> - les figures deviennent des **tables** et des **vignettes ASCII** : une image 8×8 est rendue par une ligne de caractères par rangée de pixels (`#` dense, `+` moyen, `.` faible, espace vide) ;
> - les lignes affichées par le code sont reprises **sans accents, à l'identique de la source** : ce sont des témoins de comparaison entre les deux carnets ; les tirets cadratins de la source y sont rendus par `-` ;
> - la section 7 de la source (t-SNE et UMAP) **n'est pas portée** : elle est déclarée en son lieu, plutôt que remplacée par un algorithme approché ;
> - le dispositif d'auto-évaluation `question()` de la série **n'est pas porté** : il n'a pas d'équivalent Rust, et l'auto-évaluation reste dans le carnet Python. Les cellules de vérification `question(` de la source ne se retrouvent donc pas ici.

## Introduction

Tous les carnets précédents (2.1 à 2.5) relevaient de l'apprentissage **supervisé** : on disposait d'étiquettes `y` à prédire. Ce carnet aborde l'apprentissage **non supervisé** : **aucune étiquette** n'est fournie à l'algorithme. Nous étudions deux techniques canoniques : le **clustering** (regrouper les observations similaires, ici avec KMeans) et la **réduction de dimension** (projeter des données à forte dimension dans un espace plus petit tout en préservant l'information, ici l'Analyse en Composantes Principales — ACP / PCA).

Nous travaillons sur les chiffres manuscrits de `load_digits` : 64 pixels par image (une image 8×8 aplanie en un vecteur de dimension 64) et 10 classes de chiffres (0 à 9). Les algorithmes ne verront **jamais** les étiquettes — nous les utiliserons uniquement pour **vérifier** que la structure non supervisée retrouve les classes connues.

### Objectifs d'apprentissage

À la fin de ce carnet, vous saurez :

1. Appliquer KMeans et choisir le nombre de clusters (méthode du coude) ;
2. Comprendre l'inertie (somme des carrés des écarts intra-cluster) — et la recalculer à la main, puisque `smartcore` ne publie ni les centroïdes ni la distorsion ;
3. Réduire 64 dimensions à 2 avec l'ACP, et mesurer la part de variance conservée quand la bibliothèque ne fournit pas `explained_variance_ratio_` ;
4. Interpréter ces deux grandeurs pour décider combien de clusters et combien de composantes garder, et le coût de chaque choix (exercices 1 et 2) ;
5. Reconstruire une image à partir de ses composantes principales, ce que `smartcore` ne fait pas non plus : la recomposition `mu + z · V` s'écrit dans le carnet ;
6. Reconnaître les formes où KMeans échoue par construction (demi-lunes, anneaux) et lui opposer la densité (DBSCAN) puis la hiérarchie (exercices 3, 5, 6, 7).

### Prérequis

- Carnet 2.1 (workflow ML, surapprentissage).
- Carnet 2.4 (frontière de décision, variance).
- Notions de distance euclidienne et de moyenne ; rien d'autre.

> **Référence.** MacQueen, J. (1967), *Some Methods for Classification and Analysis of Multivariate Observations*, Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability 1:281-297. — La méthode des k-moyennes et la minimisation de l'inertie intra-cluster. Voir aussi Pearson (1901) pour l'ACP, cité à la section 4.

In [2]:
// Configuration du jumeau Rust du notebook 2.6 : smartcore 0.7.0, feature datasets.
// La dependance se declare en tete de carnet, une seule fois : la crate n'est
// compilee qu'a la premiere execution.
:dep smartcore = { version = "0.7.0", features = ["datasets"] }

println!("Configuration OK : 2.6 - Clustering (KMeans) et ACP");

Configuration OK : 2.6 - Clustering (KMeans) et ACP


## 1. Le jeu de données : chiffres manuscrits

Le jeu de données `load_digits` contient 1797 images de chiffres manuscrits (0 à 9), chacune codée sur une grille de 8×8 = 64 pixels niveaux de gris. Chaque image, une fois aplanie, devient un vecteur de dimension 64 : c'est ce vecteur que les algorithmes manipulent. Les 10 classes (les chiffres 0 à 9) sont **connues** via les étiquettes `y`, mais le clustering et la PCA les **ignoreront** — nous n'utiliserons `y` qu'après coup pour vérifier que la structure non supervisée retrouve ces classes.

> **Note.** Dans `smartcore`, le jeu est exposé par `Dataset` : une structure **plate**, un `Vec<f32>` en ordre ligne (`num_samples × num_features`) et un `Vec<f32>` d'étiquettes. La cellule suivante reconstruit les lignes du jeu (une par échantillon) et convertit en `f64`, pour retrouver l'arithmétique de la source (les valeurs sont de petits entiers, 0 à 16, la conversion est donc exacte).

In [3]:
use smartcore::dataset::digits;
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;

// Le Dataset de smartcore est plat, en ordre ligne : on reconstruit les 1797 lignes 8 x 8.
fn charger_digits() -> (DenseMatrix<f64>, Vec<u32>) {
    let ds = digits::load_dataset();
    let lignes: Vec<Vec<f64>> = ds
        .data
        .chunks(ds.num_features)
        .map(|c| c.iter().map(|v| *v as f64).collect())
        .collect();
    let y: Vec<u32> = ds.target.iter().map(|v| *v as u32).collect();
    (DenseMatrix::from_2d_vec(&lignes).unwrap(), y)
}

// Rendu texte d'une image 8 x 8 : '#' pour un pixel dense, ' ' pour un pixel vide.
fn vignette(ligne: &[f64]) -> String {
    let mut s = String::new();
    for r in 0..8 {
        for c in 0..8 {
            let v = ligne[r * 8 + c];
            s.push(if v >= 8.0 {
                '#'
            } else if v >= 4.0 {
                '+'
            } else if v >= 1.0 {
                '.'
            } else {
                ' '
            });
        }
        s.push('\n');
    }
    s
}

let (x, y) = charger_digits();
let (n, p) = x.shape();

println!("X shape : ({}, {})", n, p);

let mut classes = y.clone();
classes.sort_unstable();
classes.dedup();
println!("Nombre de classes : {}", classes.len());

// Apercu du premier chiffre, en texte (la figure 2 x 5 de la source devient une vignette).
let premiere: Vec<f64> = (0..p).map(|j| *x.get((0, j))).collect();
println!("Apercu -- premiere image (label={}) :", y[0]);
print!("{}", vignette(&premiere));

X shape : (1797, 64)


Nombre de classes : 10


Apercu -- premiere image (label=0) :


  +##.  


  ####+ 


 .#. ## 


 +#  ## 


 +#  ## 


 +# .#+ 


 .#+##  


  +##   


### Lecture du jeu de données

**1797 images de 64 pixels** : chaque chiffre manuscrit est une grille 8×8 aplatie en un vecteur de 64 intensités de gris — c'est ce que dit le `(1797, 64)` affiché, et pourquoi chaque ligne de X vit dans un espace à 64 dimensions. Les 10 classes (chiffres 0 à 9) sont à peu près équilibrées, environ 180 images chacune. Le point clé pour la suite : ces étiquettes existent dans le jeu de données, mais **le clustering ne les utilisera jamais** — elles ne serviront qu'à vérifier, a posteriori, si les groupes trouvés sans supervision coïncident avec les vrais chiffres. C'est ce protocole (apprendre sans les labels, évaluer avec) qui rend ce jeu de données un banc d'essai honnête du non-supervisé.

La vignette remplace la planche de dix images de la source : elle donne la même information de contrôle (le pixel allumé dessine bien un chiffre) sans passer par un rendu graphique.

## 2. Clustering avec KMeans

KMeans partitionne les données en `k` clusters. Il procède itérativement : (1) il affecte chaque point au centroïde le plus proche, puis (2) déplace chaque centroïde sur la moyenne de ses points. L'algorithme minimise l'**inertie** (somme des carrés des distances de chaque point à son centroïde). Il faut fournir `k` en entrée — nous utilisons 10 ici (nous savons qu'il y a 10 chiffres), puis nous verrons comment choisir `k` objectivement avec la méthode du coude.

Deux écarts avec la source, tous deux déclarés :

- **L'inertie est recalculée** depuis les étiquettes rendues par `predict`, parce que les champs `centroids` et `_distortion` de `KMeans` sont privés dans `smartcore` 0.7.0. Le calcul (moyenne par cluster, puis somme des carrés des écarts) est écrit dans la cellule ; c'est la même grandeur que `kmeans.inertia_`.
- **La graine est explicite** (`seed = 42`) : `smartcore` initialise par k-means++ à partir d'un générateur semé, ce qui rend les sorties reproductibles d'une exécution à l'autre. scikit-learn, lui, fait dix initialisations (`n_init=10`) et garde la meilleure ; sans `n_init`, l'optimum local atteint dépend de l'initialisation, donc l'inertie obtenue ici peut différer de quelques pour cent de celle de la source.

In [4]:
use smartcore::dataset::digits;
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;

// Le Dataset de smartcore est plat, en ordre ligne : on reconstruit les 1797 lignes 8 x 8.
fn charger_digits() -> (DenseMatrix<f64>, Vec<u32>) {
    let ds = digits::load_dataset();
    let lignes: Vec<Vec<f64>> = ds
        .data
        .chunks(ds.num_features)
        .map(|c| c.iter().map(|v| *v as f64).collect())
        .collect();
    let y: Vec<u32> = ds.target.iter().map(|v| *v as u32).collect();
    (DenseMatrix::from_2d_vec(&lignes).unwrap(), y)
}

use smartcore::cluster::kmeans::{KMeans, KMeansParameters};

// L'inertie se recalcule depuis les etiquettes : les champs centroids et _distortion
// de KMeans sont prives.
fn inertie(x: &DenseMatrix<f64>, labels: &[u32], k: usize) -> f64 {
    let (n, p) = x.shape();
    let mut sommes = vec![0.0_f64; k * p];
    let mut tailles = vec![0usize; k];
    for i in 0..n {
        let c = labels[i] as usize;
        tailles[c] += 1;
        for j in 0..p {
            sommes[c * p + j] += *x.get((i, j));
        }
    }
    let mut total = 0.0_f64;
    for i in 0..n {
        let c = labels[i] as usize;
        let t = tailles[c].max(1) as f64;
        for j in 0..p {
            let d = *x.get((i, j)) - sommes[c * p + j] / t;
            total += d * d;
        }
    }
    total
}

// k = 1 n'est pas accepte par KMeans (il exige k >= 2) : son inertie est la variance
// totale autour du centre global.
fn inertie_k1(x: &DenseMatrix<f64>) -> f64 {
    let (n, p) = x.shape();
    let mut moy = vec![0.0_f64; p];
    for i in 0..n {
        for j in 0..p {
            moy[j] += *x.get((i, j));
        }
    }
    for j in 0..p {
        moy[j] /= n as f64;
    }
    let mut total = 0.0_f64;
    for i in 0..n {
        for j in 0..p {
            let d = *x.get((i, j)) - moy[j];
            total += d * d;
        }
    }
    total
}

// Graine explicite : l'initialisation k-means++ tire ses centres d'un generateur seme,
// ce qui rend les sorties reproductibles d'une execution a l'autre.
fn kmeans_lab(x: &DenseMatrix<f64>, k: usize) -> Vec<u32> {
    let mut params = KMeansParameters::default().with_k(k).with_max_iter(100);
    params.seed = Some(42);
    KMeans::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(x, params)
        .unwrap()
        .predict(x)
        .unwrap()
}

let (x, _y) = charger_digits();
let labels = kmeans_lab(&x, 10);
let apercu: Vec<String> = labels.iter().take(10).map(|v| v.to_string()).collect();

println!(
    "Inertie (within-cluster sum of squares) : {:.2}",
    inertie(&x, &labels, 10)
);
println!(
    "Labels de cluster (10 premiers points) : [{}]",
    apercu.join(" ")
);
println!(
    "Attention : les identifiants de cluster sont ARBITRAIRES \
     (le cluster 3 ne correspond pas forcement au chiffre 3)."
);

Inertie (within-cluster sum of squares) : 1168010.33


Labels de cluster (10 premiers points) : [0 5 5 1 8 1 2 4 7 1]


Attention : les identifiants de cluster sont ARBITRAIRES (le cluster 3 ne correspond pas forcement au chiffre 3).


### Lecture de la première expérience KMeans

L'inertie affichée est la somme des carrés des distances de chaque point au centre de son cluster : c'est la quantité que KMeans minimise, une mesure de compacité interne — plus elle est basse, plus les boules sont serrées.

Le détail affiché juste après est le piège classique du non-supervisé : les labels de cluster sont **arbitraires**. Le cluster « 3 » de la sortie n'a aucun lien avec le chiffre 3 — l'algorithme numérote ses groupes selon l'ordre d'initialisation, pas selon notre sémantique. Toute correspondance cluster → chiffre demandera une étape d'alignement explicite (ce que fera la table de contingence de la section 5).

Enfin, demander k=10 est déjà une hypothèse forte : nous savons qu'il y a 10 chiffres, mais en vraie donnée sans étiquette, choisir k serait tout le problème — c'est l'objet de la méthode du coude qui suit.

> **Comparaison des deux carnets.** L'inertie n'est pas un nombre reproductible au centième : elle dépend de l'initialisation. Les deux carnets la comparent sous une tolérance relative de 5 %, et les étiquettes affichées ne se comparent **pas** — leur ordre est arbitraire des deux côtés.

## 3. Choisir k : la méthode du coude

L'inertie **décroît** toujours quand `k` augmente (plus de clusters = des groupes plus serrés). Il n'existe pas de « vrai » `k` déductible de la seule inertie, mais la courbe présente souvent un **coude** (en anglais *elbow*) : un point d'inflexion où ajouter des clusters n'apporte plus guère de gain. Ce coude est un choix raisonnable pour `k`.

> **Note.** La source balaie `k = 1..15` ; `smartcore` refuse `k < 2` (`invalid number of clusters`). Le cas `k = 1` n'est pourtant pas une erreur : son inertie est la **variance totale autour du centre global**, et se calcule directement. La table commence donc par cette valeur calculée à la main, puis passe la main au modèle pour `k = 2..15`.

In [5]:
use smartcore::dataset::digits;
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;

// Le Dataset de smartcore est plat, en ordre ligne : on reconstruit les 1797 lignes 8 x 8.
fn charger_digits() -> (DenseMatrix<f64>, Vec<u32>) {
    let ds = digits::load_dataset();
    let lignes: Vec<Vec<f64>> = ds
        .data
        .chunks(ds.num_features)
        .map(|c| c.iter().map(|v| *v as f64).collect())
        .collect();
    let y: Vec<u32> = ds.target.iter().map(|v| *v as u32).collect();
    (DenseMatrix::from_2d_vec(&lignes).unwrap(), y)
}

use smartcore::cluster::kmeans::{KMeans, KMeansParameters};

// L'inertie se recalcule depuis les etiquettes : les champs centroids et _distortion
// de KMeans sont prives.
fn inertie(x: &DenseMatrix<f64>, labels: &[u32], k: usize) -> f64 {
    let (n, p) = x.shape();
    let mut sommes = vec![0.0_f64; k * p];
    let mut tailles = vec![0usize; k];
    for i in 0..n {
        let c = labels[i] as usize;
        tailles[c] += 1;
        for j in 0..p {
            sommes[c * p + j] += *x.get((i, j));
        }
    }
    let mut total = 0.0_f64;
    for i in 0..n {
        let c = labels[i] as usize;
        let t = tailles[c].max(1) as f64;
        for j in 0..p {
            let d = *x.get((i, j)) - sommes[c * p + j] / t;
            total += d * d;
        }
    }
    total
}

// k = 1 n'est pas accepte par KMeans (il exige k >= 2) : son inertie est la variance
// totale autour du centre global.
fn inertie_k1(x: &DenseMatrix<f64>) -> f64 {
    let (n, p) = x.shape();
    let mut moy = vec![0.0_f64; p];
    for i in 0..n {
        for j in 0..p {
            moy[j] += *x.get((i, j));
        }
    }
    for j in 0..p {
        moy[j] /= n as f64;
    }
    let mut total = 0.0_f64;
    for i in 0..n {
        for j in 0..p {
            let d = *x.get((i, j)) - moy[j];
            total += d * d;
        }
    }
    total
}

// Graine explicite : l'initialisation k-means++ tire ses centres d'un generateur seme,
// ce qui rend les sorties reproductibles d'une execution a l'autre.
fn kmeans_lab(x: &DenseMatrix<f64>, k: usize) -> Vec<u32> {
    let mut params = KMeansParameters::default().with_k(k).with_max_iter(100);
    params.seed = Some(42);
    KMeans::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(x, params)
        .unwrap()
        .predict(x)
        .unwrap()
}

let (x, _y) = charger_digits();
let mut inerties = vec![inertie_k1(&x)];
for k in 2..=15 {
    let lab = kmeans_lab(&x, k);
    inerties.push(inertie(&x, &lab, k));
}

println!("{:>3}  {:>14}  {:>14}", "k", "inertie", "gain");
for (i, v) in inerties.iter().enumerate() {
    let gain = if i == 0 { 0.0 } else { inerties[i - 1] - v };
    println!("{:>3}  {:>14.2}  {:>14.2}", i + 1, v, gain);
}

println!("Inertie pour k=10 : {:.2}", inerties[9]);
println!("Un coude est visible autour de k=10, coherent avec les 10 chiffres.");

  k         inertie            gain


  1      2159057.29            0.00


  2      1914619.62       244437.67


  3      1733042.57       181577.05


  4      1612274.31       120768.26


  5      1498830.89       113443.42


... (8 outputs left out)

 14      1053488.42        30254.83


 15      1043880.58         9607.84


Inertie pour k=10 : 1168010.33


Un coude est visible autour de k=10, coherent avec les 10 chiffres.


### Lecture de la courbe du coude

La table joue ici le rôle de la courbe de la source : colonne `inertie`, et colonne `gain` qui donne la baisse obtenue en ajoutant un cluster. L'inertie décroît vite puis s'aplatit — chaque cluster ajouté retire de l'inertie, avec un rendement décroissant, et c'est ce rendement que la colonne `gain` rend lisible sans graphique.

Le **coude** est le point où le gain marginal tombe franchement : payer un cluster de plus ne « serre » presque plus rien. La colonne `gain` le situe sans ambiguïté — elle vaut 68 226 au passage à k=9, puis **34 304** au passage à k=10, et reste ensuite autour de 30 000 (15 359 à k=13, 9 608 à k=15). Le rendement est donc divisé par deux précisément au dixième cluster. Que l'algorithme, qui n'a jamais vu les étiquettes, place ce décrochage sur le nombre de chiffres connus est le message central du non-supervisé : la classe n'est pas une annotation magique, c'est une régularité de la donnée que la compacité seule suffit à révéler — au prix, ici, d'un œil sur la table pour trancher où s'arrêter.

> **Ce que la table ne montre pas.** La source trace une figure ; ce carnet affiche des nombres. Un coude se lit aussi bien sur une colonne de gains, mais il reste un **jugement** : deux lecteurs peuvent placer le coude à 8 ou à 12. C'est précisément pourquoi l'exercice 1 demande de chiffrer la chute plutôt que de désigner un point.

### Exercice 1 : la décroissance de l'inertie

**Objectif** : mesurer la chute d'inertie lorsqu'on passe de k=5 à k=10 clusters, à partir de la liste `inerties`.

**Indice** : `inerties` est une liste 0-indexée (indice 0 = k=1, indice 4 = k=5, indice 9 = k=10). La baisse vaut `inerties[4] - inerties[9]`.

La cellule recalcule la liste (elle est autonome) puis affiche les deux valeurs ; il ne reste qu'à remplacer `None` par l'expression. Le stub ne lève pas d'erreur : il affiche ce qui manque.

In [6]:
use smartcore::dataset::digits;
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;

// Le Dataset de smartcore est plat, en ordre ligne : on reconstruit les 1797 lignes 8 x 8.
fn charger_digits() -> (DenseMatrix<f64>, Vec<u32>) {
    let ds = digits::load_dataset();
    let lignes: Vec<Vec<f64>> = ds
        .data
        .chunks(ds.num_features)
        .map(|c| c.iter().map(|v| *v as f64).collect())
        .collect();
    let y: Vec<u32> = ds.target.iter().map(|v| *v as u32).collect();
    (DenseMatrix::from_2d_vec(&lignes).unwrap(), y)
}

use smartcore::cluster::kmeans::{KMeans, KMeansParameters};

// L'inertie se recalcule depuis les etiquettes : les champs centroids et _distortion
// de KMeans sont prives.
fn inertie(x: &DenseMatrix<f64>, labels: &[u32], k: usize) -> f64 {
    let (n, p) = x.shape();
    let mut sommes = vec![0.0_f64; k * p];
    let mut tailles = vec![0usize; k];
    for i in 0..n {
        let c = labels[i] as usize;
        tailles[c] += 1;
        for j in 0..p {
            sommes[c * p + j] += *x.get((i, j));
        }
    }
    let mut total = 0.0_f64;
    for i in 0..n {
        let c = labels[i] as usize;
        let t = tailles[c].max(1) as f64;
        for j in 0..p {
            let d = *x.get((i, j)) - sommes[c * p + j] / t;
            total += d * d;
        }
    }
    total
}

// k = 1 n'est pas accepte par KMeans (il exige k >= 2) : son inertie est la variance
// totale autour du centre global.
fn inertie_k1(x: &DenseMatrix<f64>) -> f64 {
    let (n, p) = x.shape();
    let mut moy = vec![0.0_f64; p];
    for i in 0..n {
        for j in 0..p {
            moy[j] += *x.get((i, j));
        }
    }
    for j in 0..p {
        moy[j] /= n as f64;
    }
    let mut total = 0.0_f64;
    for i in 0..n {
        for j in 0..p {
            let d = *x.get((i, j)) - moy[j];
            total += d * d;
        }
    }
    total
}

// Graine explicite : l'initialisation k-means++ tire ses centres d'un generateur seme,
// ce qui rend les sorties reproductibles d'une execution a l'autre.
fn kmeans_lab(x: &DenseMatrix<f64>, k: usize) -> Vec<u32> {
    let mut params = KMeansParameters::default().with_k(k).with_max_iter(100);
    params.seed = Some(42);
    KMeans::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(x, params)
        .unwrap()
        .predict(x)
        .unwrap()
}

let (x, _y) = charger_digits();
let mut inerties = vec![inertie_k1(&x)];
for k in 2..=10 {
    let lab = kmeans_lab(&x, k);
    inerties.push(inertie(&x, &lab, k));
}

println!(
    "inerties[4] (k=5) = {:.2} ; inerties[9] (k=10) = {:.2}",
    inerties[4], inerties[9]
);

// TODO etudiant : remplacer None par inerties[4] - inerties[9]
let baisse_inertie: Option<f64> = None;

match baisse_inertie {
    Some(v) => println!("Exercice 1 : la baisse d'inertie de k=5 a k=10 vaut {:.2}", v),
    None => println!(
        "Exercice 1 a completer : baisse_inertie vaut None (indice : inerties[4] - inerties[9])."
    ),
}

inerties[4] (k=5) = 1498830.89 ; inerties[9] (k=10) = 1168010.33


Exercice 1 a completer : baisse_inertie vaut None (indice : inerties[4] - inerties[9]).


()

## 4. Réduction de dimension : l'ACP (PCA)

Chaque chiffre est un point dans un espace à 64 dimensions — impossible à tracer directement. La PCA (*Principal Component Analysis*, ou ACP en français) identifie les directions de **variance maximale** et projette les données sur les premières. La première composante principale capture le plus de variance, la deuxième le résidu orthogonal, etc. Projeter sur les 2 premières composantes donne un nuage de points 2D que l'on PEUT visualiser — et souvent les classes s'y séparent.

> **Référence.** Pearson, K. (1901), *On Lines and Planes of Closest Fit to Systems of Points in Space*, Philosophical Magazine 2(11):559-572. Article fondateur de l'Analyse en Composantes Principales : la recherche des axes de variance maximale comme projection optimale (au sens des moindres carrés).

> **Note.** `PCA` de `smartcore` n'expose ni `explained_variance_ratio_`, ni les valeurs propres, ni `inverse_transform` : `transform` rend `(x − mu) · V` et `components()` rend `V` (de forme 64 × nombre de composantes — l'inverse de `components_` de scikit-learn). La part de variance expliquée se recalcule donc en deux lignes : variance des colonnes projetées, rapportée à la variance totale. Comme `transform` ne remet pas à l'échelle, le rapport des variances égale le rapport des valeurs propres, sans facteur à corriger.

In [7]:
use smartcore::dataset::digits;
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;

// Le Dataset de smartcore est plat, en ordre ligne : on reconstruit les 1797 lignes 8 x 8.
fn charger_digits() -> (DenseMatrix<f64>, Vec<u32>) {
    let ds = digits::load_dataset();
    let lignes: Vec<Vec<f64>> = ds
        .data
        .chunks(ds.num_features)
        .map(|c| c.iter().map(|v| *v as f64).collect())
        .collect();
    let y: Vec<u32> = ds.target.iter().map(|v| *v as u32).collect();
    (DenseMatrix::from_2d_vec(&lignes).unwrap(), y)
}

use smartcore::decomposition::pca::{PCA, PCAParameters};

fn moyenne_colonnes(x: &DenseMatrix<f64>) -> Vec<f64> {
    let (n, p) = x.shape();
    let mut moy = vec![0.0_f64; p];
    for i in 0..n {
        for j in 0..p {
            moy[j] += *x.get((i, j));
        }
    }
    for j in 0..p {
        moy[j] /= n as f64;
    }
    moy
}

fn variances_colonnes(x: &DenseMatrix<f64>) -> Vec<f64> {
    let (n, p) = x.shape();
    let moy = moyenne_colonnes(x);
    let mut var = vec![0.0_f64; p];
    for i in 0..n {
        for j in 0..p {
            let d = *x.get((i, j)) - moy[j];
            var[j] += d * d;
        }
    }
    for j in 0..p {
        var[j] /= (n - 1) as f64;
    }
    var
}

let (x, y) = charger_digits();
let pca = PCA::<f64, DenseMatrix<f64>>::fit(&x, PCAParameters::default().with_n_components(2))
    .unwrap();
let z = pca.transform(&x).unwrap();

let totale: f64 = variances_colonnes(&x).iter().sum();
let projete: f64 = variances_colonnes(&z).iter().sum();
let variance_2d = projete / totale;

println!(
    "Variance expliquee par les 2 premieres composantes : {:.3} ({:.1}%)",
    variance_2d,
    variance_2d * 100.0
);

// Temoin mesurable a la place de la figure : part de la variance du plan 2D portee par
// les 10 classes. Sans structure, cette part vaudrait environ 1/10.
let (n, _) = z.shape();
let mut tailles = vec![0usize; 10];
let mut sommes = vec![0.0_f64; 20];
for i in 0..n {
    let c = y[i] as usize;
    tailles[c] += 1;
    sommes[c * 2] += *z.get((i, 0));
    sommes[c * 2 + 1] += *z.get((i, 1));
}
let mut sst = 0.0_f64;
for i in 0..n {
    for c in 0..2 {
        let d = *z.get((i, c));
        sst += d * d;
    }
}
let mut ssb = 0.0_f64;
for c in 0..10 {
    let t = tailles[c].max(1) as f64;
    let m0 = sommes[c * 2] / t;
    let m1 = sommes[c * 2 + 1] / t;
    ssb += t * (m0 * m0 + m1 * m1);
}
println!(
    "Part de la variance du plan 2D portee par les 10 classes : {:.3}",
    ssb / sst
);
println!(
    "Meme en 2D, les classes de chiffres forment des groupes visibles : \
     la PCA non supervisee retrouve la structure connue."
);

Variance expliquee par les 2 premieres composantes : 0.285 (28.5%)


Part de la variance du plan 2D portee par les 10 classes : 0.741


Meme en 2D, les classes de chiffres forment des groupes visibles : la PCA non supervisee retrouve la structure connue.


### Lecture de la projection ACP

Les deux premières composantes ne capturent que **28,5 %** de la variance — la même valeur que la source, au chiffre près : projeter 64 dimensions sur 2 en jette les sept dixièmes. Et pourtant le nuage montre déjà des regroupements — les classes les plus séparées dans l'espace original restent séparées après projection. C'est le compromis de l'ACP : c'est la **meilleure** projection linéaire possible (au sens de la variance conservée), mais elle reste linéaire — deux chiffres séparés par une frontière courbe dans l'espace original peuvent se chevaucher une fois aplatis.

Le second nombre remplace la lecture de la figure : la **part de la variance du plan 2D portée par les dix classes**, mesurée à **0,741**. Une projection qui ne mélangerait pas les chiffres y laisserait presque toute la variance (proche de 1) ; une projection qui les mélangerait entièrement y laisserait environ un dixième, la part attendue de dix groupes de même taille sans structure. Sept fois le hasard, mais pas l'unité : la PCA sépare **largement** sans séparer **parfaitement**. C'est exactement le constat que la figure de la source laissait à l'œil, et c'est lui qui motive sa section 7 — les variations non linéaires qu'aucun plan ne peut démêler.

L'exigence des 95 % de l'exercice 2 mesurera exactement ce que coûte la compression : combien de directions faut-il garder pour ne presque rien perdre.

### Exercice 2 : combien de composantes garder ?

Les 2 premières composantes ne captent qu'une partie de la variance. Pour compresser sans trop perdre d'information, on cherche le nombre de composantes suffisant à atteindre **95 % de variance cumulée**.

**Étapes**

1. Ajustez une ACP sur `x` en gardant **toutes** les composantes (`with_n_components(64)` : le nombre d'attributs), puis projetez. La `pca` de la section 4 n'en garde que 2, il faut donc un nouvel ajustement.
2. Le cumul se calcule sur la variance de chaque colonne projetée, rapportée à la variance totale de `x` : c'est ce que fait déjà la cellule (elle affiche la cumulative à quelques nombres de composantes).
3. Trouvez le premier indice où la cumulée dépasse 0.95, et **ajoutez 1** pour passer de l'indice au nombre de composantes.

**Indice** : la cellule affiche déjà `cumul[0], cumul[1], cumul[4], ...` — de quoi situer la réponse avant de la calculer. `None` ne lève pas d'erreur : le stub affiche ce qui manque.

In [8]:
use smartcore::dataset::digits;
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;

// Le Dataset de smartcore est plat, en ordre ligne : on reconstruit les 1797 lignes 8 x 8.
fn charger_digits() -> (DenseMatrix<f64>, Vec<u32>) {
    let ds = digits::load_dataset();
    let lignes: Vec<Vec<f64>> = ds
        .data
        .chunks(ds.num_features)
        .map(|c| c.iter().map(|v| *v as f64).collect())
        .collect();
    let y: Vec<u32> = ds.target.iter().map(|v| *v as u32).collect();
    (DenseMatrix::from_2d_vec(&lignes).unwrap(), y)
}

use smartcore::decomposition::pca::{PCA, PCAParameters};

fn moyenne_colonnes(x: &DenseMatrix<f64>) -> Vec<f64> {
    let (n, p) = x.shape();
    let mut moy = vec![0.0_f64; p];
    for i in 0..n {
        for j in 0..p {
            moy[j] += *x.get((i, j));
        }
    }
    for j in 0..p {
        moy[j] /= n as f64;
    }
    moy
}

fn variances_colonnes(x: &DenseMatrix<f64>) -> Vec<f64> {
    let (n, p) = x.shape();
    let moy = moyenne_colonnes(x);
    let mut var = vec![0.0_f64; p];
    for i in 0..n {
        for j in 0..p {
            let d = *x.get((i, j)) - moy[j];
            var[j] += d * d;
        }
    }
    for j in 0..p {
        var[j] /= (n - 1) as f64;
    }
    var
}

let (x, _y) = charger_digits();
let totale: f64 = variances_colonnes(&x).iter().sum();
let pca64 = PCA::<f64, DenseMatrix<f64>>::fit(&x, PCAParameters::default().with_n_components(64))
    .unwrap();
let z64 = pca64.transform(&x).unwrap();

let mut cumul: Vec<f64> = Vec::with_capacity(64);
let mut s = 0.0_f64;
for v in variances_colonnes(&z64) {
    s += v;
    cumul.push(s / totale);
}

for n_comp in [1, 2, 5, 10, 20, 30, 40, 50, 64] {
    println!(
        "  {:>2} composantes : variance cumulee = {:.3}",
        n_comp,
        cumul[n_comp - 1]
    );
}

// TODO etudiant : premier indice ou la cumulee depasse 0.95, puis ajouter 1
let n_composantes_95: Option<usize> = None;

match n_composantes_95 {
    Some(n) => println!(
        "Exercice 2 : {} composantes suffisent pour 95% de la variance (cumul a cet indice : {:.3}).",
        n,
        cumul[n - 1]
    ),
    None => println!(
        "Exercice 2 a completer : n_composantes_95 vaut None \
         (indice : premier indice de cumul qui depasse 0.95, plus 1)."
    ),
}

   1 composantes : variance cumulee = 0.149


   2 composantes : variance cumulee = 0.285


   5 composantes : variance cumulee = 0.545


  10 composantes : variance cumulee = 0.738


  20 composantes : variance cumulee = 0.894


  30 composantes : variance cumulee = 0.959


  40 composantes : variance cumulee = 0.988


  50 composantes : variance cumulee = 1.000


  64 composantes : variance cumulee = 1.000


Exercice 2 a completer : n_composantes_95 vaut None (indice : premier indice de cumul qui depasse 0.95, plus 1).


()

## 5. Vérifier le clustering contre les vraies étiquettes

KMeans a trouvé 10 clusters **sans voir les étiquettes**. Ces clusters correspondent-ils aux 10 classes de chiffres ? On peut le vérifier avec une matrice de confusion entre `kmeans.labels_` et `y`. **Attention** : les identifiants de cluster étant arbitraires, la matrice est une permutation des colonnes — mais si une structure diagonale se dessine, la méthode non supervisée a bien retrouvé les classes. C'est la démonstration honnête que l'apprentissage non supervisé redécouvre une structure connue.

La source affiche cette matrice en `imshow`. Sans rendu graphique, le carnet en donne la **table de contingence** complète : dix lignes (vrai chiffre), dix colonnes (cluster), la colonne `dominant` donnant le cluster majoritaire de chaque ligne. Une table porte une information que l'image ne porte pas — les effectifs exacts.

In [9]:
use smartcore::dataset::digits;
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;

// Le Dataset de smartcore est plat, en ordre ligne : on reconstruit les 1797 lignes 8 x 8.
fn charger_digits() -> (DenseMatrix<f64>, Vec<u32>) {
    let ds = digits::load_dataset();
    let lignes: Vec<Vec<f64>> = ds
        .data
        .chunks(ds.num_features)
        .map(|c| c.iter().map(|v| *v as f64).collect())
        .collect();
    let y: Vec<u32> = ds.target.iter().map(|v| *v as u32).collect();
    (DenseMatrix::from_2d_vec(&lignes).unwrap(), y)
}

use smartcore::cluster::kmeans::{KMeans, KMeansParameters};

// L'inertie se recalcule depuis les etiquettes : les champs centroids et _distortion
// de KMeans sont prives.
fn inertie(x: &DenseMatrix<f64>, labels: &[u32], k: usize) -> f64 {
    let (n, p) = x.shape();
    let mut sommes = vec![0.0_f64; k * p];
    let mut tailles = vec![0usize; k];
    for i in 0..n {
        let c = labels[i] as usize;
        tailles[c] += 1;
        for j in 0..p {
            sommes[c * p + j] += *x.get((i, j));
        }
    }
    let mut total = 0.0_f64;
    for i in 0..n {
        let c = labels[i] as usize;
        let t = tailles[c].max(1) as f64;
        for j in 0..p {
            let d = *x.get((i, j)) - sommes[c * p + j] / t;
            total += d * d;
        }
    }
    total
}

// k = 1 n'est pas accepte par KMeans (il exige k >= 2) : son inertie est la variance
// totale autour du centre global.
fn inertie_k1(x: &DenseMatrix<f64>) -> f64 {
    let (n, p) = x.shape();
    let mut moy = vec![0.0_f64; p];
    for i in 0..n {
        for j in 0..p {
            moy[j] += *x.get((i, j));
        }
    }
    for j in 0..p {
        moy[j] /= n as f64;
    }
    let mut total = 0.0_f64;
    for i in 0..n {
        for j in 0..p {
            let d = *x.get((i, j)) - moy[j];
            total += d * d;
        }
    }
    total
}

// Graine explicite : l'initialisation k-means++ tire ses centres d'un generateur seme,
// ce qui rend les sorties reproductibles d'une execution a l'autre.
fn kmeans_lab(x: &DenseMatrix<f64>, k: usize) -> Vec<u32> {
    let mut params = KMeansParameters::default().with_k(k).with_max_iter(100);
    params.seed = Some(42);
    KMeans::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(x, params)
        .unwrap()
        .predict(x)
        .unwrap()
}

let (x, y) = charger_digits();
let labels = kmeans_lab(&x, 10);

let mut table = vec![vec![0usize; 10]; 10];
for i in 0..y.len() {
    table[y[i] as usize][labels[i] as usize] += 1;
}

println!("Chaque ligne = un vrai chiffre, chaque colonne = un cluster.");
println!("Une structure diagonale (a permutation des colonnes pres) signifie");
println!("que les clusters correspondent aux classes de chiffres.");
println!();

print!("chiffre |");
for j in 0..10 {
    print!("{:>5}", j);
}
println!("  | total | dominant");

for i in 0..10 {
    print!("{:>7} |", i);
    let mut total = 0usize;
    let mut meilleur = 0usize;
    for j in 0..10 {
        print!("{:>5}", table[i][j]);
        total += table[i][j];
        if table[i][j] > table[i][meilleur] {
            meilleur = j;
        }
    }
    println!("  |{:>6} | {:>8}", total, meilleur);
}

Chaque ligne = un vrai chiffre, chaque colonne = un cluster.


Une structure diagonale (a permutation des colonnes pres) signifie


que les clusters correspondent aux classes de chiffres.


chiffre |    0    1    2    3    4    5    6    7    8    9  | total | dominant


      0 |  177    0    0    0    0    0    0    0    1    0  |   178 |        0


      1 |    0    0    2    1    0  100   54    0    0   25  |   182 |        5


... (5 outputs left out)

      7 |    0    0    0    0  164    5    0   10    0    0  |   179 |        4


      8 |    0   13    1    3    2   13    6  133    0    3  |   174 |        7


      9 |    0  145    0    4    7    0   20    4    0    0  |   180 |        1


()

### Lecture du résultat — la diagonale à permutation près

La table montre une structure **quasi diagonale, à une permutation des colonnes près** : chaque ligne (vrai chiffre) concentre l'essentiel de sa masse sur une colonne (un cluster) — mais cette colonne change d'une ligne à l'autre, conséquence directe des identifiants de cluster arbitraires. Deux lectures concrètes de ce que la géométrie de l'espace pixel révèle :

1. les cases hors diagonale portant une masse non négligeable indiquent les chiffres que KMeans confond dans cet espace — des écritures dont les vecteurs de 64 pixels se recouvrent ;
2. un chiffre dont la masse se partage entre deux colonnes signale une classe coupée en deux modes d'écriture (deux « styles » du même chiffre que deux clusters distincts absorbent).

C'est une information qu'un simple taux de réussite supervisé ne donne pas : la table montre **où** la structure non supervisée échoue, pas seulement **combien**. La section 7 de la source prolonge cette lecture par une projection non linéaire, qui sépare visuellement ces recouvrements ; elle n'est pas portée ici, et la section correspondante de ce carnet dit ce qu'elle aurait montré.

### Exercice 3 : les clusters correspondent-ils aux chiffres ?

**Objectif** : pour le vrai chiffre « 0 » (les points où `y == 0`), trouver quel cluster (0 à 9) est le plus fréquent parmi ceux attribués par KMeans.

**Étapes**

1. Construisez le masque des points du chiffre 0 : `y[i] == 0`.
2. Sélectionnez les clusters de ces points (la cellule les met déjà dans `clusters_zero`).
3. Trouvez le cluster dominant : celui dont l'effectif est le plus grand dans `clusters_zero` (`np.bincount(...).argmax()` côté Python ; `compte` est fourni ici).

La cellule affiche déjà la répartition complète du chiffre 0 par cluster : la réponse s'y lit, l'exercice est de l'extraire par le calcul.

In [10]:
use smartcore::dataset::digits;
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;

// Le Dataset de smartcore est plat, en ordre ligne : on reconstruit les 1797 lignes 8 x 8.
fn charger_digits() -> (DenseMatrix<f64>, Vec<u32>) {
    let ds = digits::load_dataset();
    let lignes: Vec<Vec<f64>> = ds
        .data
        .chunks(ds.num_features)
        .map(|c| c.iter().map(|v| *v as f64).collect())
        .collect();
    let y: Vec<u32> = ds.target.iter().map(|v| *v as u32).collect();
    (DenseMatrix::from_2d_vec(&lignes).unwrap(), y)
}

use smartcore::cluster::kmeans::{KMeans, KMeansParameters};

// L'inertie se recalcule depuis les etiquettes : les champs centroids et _distortion
// de KMeans sont prives.
fn inertie(x: &DenseMatrix<f64>, labels: &[u32], k: usize) -> f64 {
    let (n, p) = x.shape();
    let mut sommes = vec![0.0_f64; k * p];
    let mut tailles = vec![0usize; k];
    for i in 0..n {
        let c = labels[i] as usize;
        tailles[c] += 1;
        for j in 0..p {
            sommes[c * p + j] += *x.get((i, j));
        }
    }
    let mut total = 0.0_f64;
    for i in 0..n {
        let c = labels[i] as usize;
        let t = tailles[c].max(1) as f64;
        for j in 0..p {
            let d = *x.get((i, j)) - sommes[c * p + j] / t;
            total += d * d;
        }
    }
    total
}

// k = 1 n'est pas accepte par KMeans (il exige k >= 2) : son inertie est la variance
// totale autour du centre global.
fn inertie_k1(x: &DenseMatrix<f64>) -> f64 {
    let (n, p) = x.shape();
    let mut moy = vec![0.0_f64; p];
    for i in 0..n {
        for j in 0..p {
            moy[j] += *x.get((i, j));
        }
    }
    for j in 0..p {
        moy[j] /= n as f64;
    }
    let mut total = 0.0_f64;
    for i in 0..n {
        for j in 0..p {
            let d = *x.get((i, j)) - moy[j];
            total += d * d;
        }
    }
    total
}

// Graine explicite : l'initialisation k-means++ tire ses centres d'un generateur seme,
// ce qui rend les sorties reproductibles d'une execution a l'autre.
fn kmeans_lab(x: &DenseMatrix<f64>, k: usize) -> Vec<u32> {
    let mut params = KMeansParameters::default().with_k(k).with_max_iter(100);
    params.seed = Some(42);
    KMeans::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(x, params)
        .unwrap()
        .predict(x)
        .unwrap()
}

fn compte(v: &[u32], c: u32) -> usize {
    v.iter().filter(|&&x| x == c).count()
}

let (x, y) = charger_digits();
let labels = kmeans_lab(&x, 10);
let clusters_zero: Vec<u32> = (0..y.len())
    .filter(|&i| y[i] == 0)
    .map(|i| labels[i])
    .collect();

println!("Le chiffre 0 est represente par {} images.", clusters_zero.len());
let mut repartition: Vec<(u32, usize)> = (0..10)
    .map(|c| (c as u32, compte(&clusters_zero, c as u32)))
    .collect();
repartition.retain(|&(_, n)| n > 0);
println!("Repartition de ces images par cluster : {:?}", repartition);

// TODO etudiant : le cluster le plus frequent parmi clusters_zero
let cluster_dominant_zero: Option<usize> = None;

match cluster_dominant_zero {
    Some(c) => println!(
        "Exercice 3 : le chiffre 0 part majoritairement dans le cluster {} ({} images sur {}).",
        c,
        compte(&clusters_zero, c as u32),
        clusters_zero.len()
    ),
    None => println!(
        "Exercice 3 a completer : cluster_dominant_zero vaut None \
         (indice : compter les occurrences de chaque cluster, garder le plus grand)."
    ),
}

Le chiffre 0 est represente par 178 images.


Repartition de ces images par cluster : [(0, 177), (8, 1)]


Exercice 3 a completer : cluster_dominant_zero vaut None (indice : compter les occurrences de chaque cluster, garder le plus grand).


()

## 6. Reconstruire une image à partir de la PCA

La PCA est aussi une **compression** : projeter sur `k` composantes puis *inverse-transformer* reconstruit une approximation de l'original. Moins de composantes = plus de compression et plus de flou. Cette section illustre le compromis entre réduction de dimension et perte d'information.

`smartcore` ne fournit pas `inverse_transform` : la recomposition s'écrit ici en clair, `mu + z · V`, où `mu` est la moyenne de colonne (recalculée sur les données, le champ `mu` de `PCA` étant privé) et `V` la matrice rendue par `components()`. L'erreur affichée est l'**erreur quadratique moyenne par pixel**, sur tout le jeu — une mesure qui décroît avec le nombre de composantes, et que la source laissait à l'œil sur six vignettes.

In [11]:
use smartcore::dataset::digits;
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;

// Le Dataset de smartcore est plat, en ordre ligne : on reconstruit les 1797 lignes 8 x 8.
fn charger_digits() -> (DenseMatrix<f64>, Vec<u32>) {
    let ds = digits::load_dataset();
    let lignes: Vec<Vec<f64>> = ds
        .data
        .chunks(ds.num_features)
        .map(|c| c.iter().map(|v| *v as f64).collect())
        .collect();
    let y: Vec<u32> = ds.target.iter().map(|v| *v as u32).collect();
    (DenseMatrix::from_2d_vec(&lignes).unwrap(), y)
}

use smartcore::decomposition::pca::{PCA, PCAParameters};

fn moyenne_colonnes(x: &DenseMatrix<f64>) -> Vec<f64> {
    let (n, p) = x.shape();
    let mut moy = vec![0.0_f64; p];
    for i in 0..n {
        for j in 0..p {
            moy[j] += *x.get((i, j));
        }
    }
    for j in 0..p {
        moy[j] /= n as f64;
    }
    moy
}

fn variances_colonnes(x: &DenseMatrix<f64>) -> Vec<f64> {
    let (n, p) = x.shape();
    let moy = moyenne_colonnes(x);
    let mut var = vec![0.0_f64; p];
    for i in 0..n {
        for j in 0..p {
            let d = *x.get((i, j)) - moy[j];
            var[j] += d * d;
        }
    }
    for j in 0..p {
        var[j] /= (n - 1) as f64;
    }
    var
}

// Rendu texte d'une image 8 x 8 : '#' pour un pixel dense, ' ' pour un pixel vide.
fn vignette(ligne: &[f64]) -> String {
    let mut s = String::new();
    for r in 0..8 {
        for c in 0..8 {
            let v = ligne[r * 8 + c];
            s.push(if v >= 8.0 {
                '#'
            } else if v >= 4.0 {
                '+'
            } else if v >= 1.0 {
                '.'
            } else {
                ' '
            });
        }
        s.push('\n');
    }
    s
}

// Recomposition mu + z * V : V = components() est de forme (64, n_comp).
fn reconstruire(
    n: usize,
    p: usize,
    z: &DenseMatrix<f64>,
    comp: &DenseMatrix<f64>,
    mu: &[f64],
) -> DenseMatrix<f64> {
    let (_, n_comp) = comp.shape();
    let mut out = vec![0.0_f64; n * p];
    for i in 0..n {
        for j in 0..p {
            let mut v = mu[j];
            for c in 0..n_comp {
                v += *z.get((i, c)) * *comp.get((j, c));
            }
            out[i * p + j] = v;
        }
    }
    DenseMatrix::new(n, p, out, false).unwrap()
}

let (x, _y) = charger_digits();
let (n, p) = x.shape();
let mu = moyenne_colonnes(&x);
let mut derniere: Option<DenseMatrix<f64>> = None;

for n_comp in [1, 5, 10, 20, 50] {
    let pca =
        PCA::<f64, DenseMatrix<f64>>::fit(&x, PCAParameters::default().with_n_components(n_comp))
            .unwrap();
    let z = pca.transform(&x).unwrap();
    let rec = reconstruire(n, p, &z, pca.components(), &mu);
    let mut err = 0.0_f64;
    for i in 0..n {
        for j in 0..p {
            let d = *x.get((i, j)) - *rec.get((i, j));
            err += d * d;
        }
    }
    println!(
        "{:>2} comp. : erreur quadratique moyenne par pixel = {:.4}",
        n_comp,
        (err / (n * p) as f64).sqrt()
    );
    derniere = Some(rec);
}

let rec = derniere.unwrap();
let original: Vec<f64> = (0..p).map(|j| *x.get((0, j))).collect();
let approx: Vec<f64> = (0..p).map(|j| *rec.get((0, j))).collect();

println!();
println!("Chiffre 0 -- original :");
print!("{}", vignette(&original));
println!("Chiffre 0 -- reconstruit avec 50 composantes :");
print!("{}", vignette(&approx));
println!("Plus on garde de composantes, plus l'image reconstruite devient nette.");

 1 comp. : erreur quadratique moyenne par pixel = 3.9972


 5 comp. : erreur quadratique moyenne par pixel = 2.9227


10 comp. : erreur quadratique moyenne par pixel = 2.2168


20 comp. : erreur quadratique moyenne par pixel = 1.4086


50 comp. : erreur quadratique moyenne par pixel = 0.0922


Chiffre 0 -- original :


... (14 outputs left out)

 +# .#+ 


 .#+##  


  +##   


Plus on garde de composantes, plus l'image reconstruite devient nette.


### Lecture de la reconstruction

L'erreur quadratique moyenne par pixel **décroît** avec le nombre de composantes, et c'est une nécessité, pas une observation : les composantes sont ordonnées par valeur propre décroissante, donc chacune apporte moins que la précédente et aucune ne peut retirer de l'information déjà captée. La forme de la décroissance, elle, se lit dans les cinq mesures : **3,9972** (1 composante), **2,9227** (5), **2,2168** (10), **1,4086** (20), **0,0922** (50). Les quatre premières lignes se suivent à pas réguliers — de 1 à 5 composantes, l'erreur ne baisse que de 27 % — et c'est le dernier pas, de 20 à 50, qui l'effondre d'un facteur **quinze**. Autrement dit : la dizaine de directions qui porte la structure visible laisse encore un résidu net, et la netteté de la vignette vient des composantes tardives, celles dont la variance est faible. C'est le même rendement décroissant que la courbe du coude, vu depuis l'autre bout — mais il ne se voit qu'en regardant où l'erreur **s'effondre**, pas où elle descend le plus vite.

Les deux vignettes donnent le contrôle visuel que la source obtenait avec six images : le chiffre 0 reconstruit à 50 composantes se lit comme l'original, la différence restant dans les pixels faibles.

> **Note.** La cellule de la source n'affiche **aucun nombre** — seulement six vignettes. L'erreur mesurée ici n'a donc pas de contrepartie Python et ne figure pas au registre de conformité ; elle est en revanche un invariant testable (décroissance stricte sur la plage) que la page de spécification déclare.

## 7. Au-delà de l'ACP : t-SNE et UMAP — non porté (déclaré)

La projection ACP de la section 4 est **linéaire** : chaque composante est une combinaison linéaire des 64 pixels. C'est une limite structurelle — deux chiffres peuvent être proches dans l'espace pixel tout en étant séparés par une variation *non linéaire* (la boucle d'un 8, la barre d'un 7), et aucun plan de projection linéaire ne peut les découper proprement. Le résultat s'est vu sur la figure de la source : les classes de chiffres se chevauchent, et la table de contingence de la section 5 dit **lesquelles**.

**t-SNE** (*t-distributed Stochastic Neighbor Embedding*, van der Maaten & Hinton 2008) attaque le problème différemment : au lieu de chercher des axes, il construit une projection 2D qui **préserve les voisinages** — deux points proches en dimension 64 doivent rester proches en 2D (et réciproquement), en mesurant la « proximité » par des probabilités gaussiennes côté haute dimension et une loi de Student côté basse dimension, puis en minimisant la divergence de Kullback-Leibler entre les deux distributions. La projection obtenue est **non linéaire par construction**.

### Pourquoi cette section n'est pas portée

Ce carnet s'arrête ici, et le dit plutôt que de le combler :

1. **Aucune crate du périmètre ne porte ces deux algorithmes.** Le lot de portage s'appuie sur `smartcore` 0.7.0 ; t-SNE et UMAP y sont absents. Les ajouter supposerait `linfa-tsne` / `linfa-umap`, c'est-à-dire un **autre moteur** que celui que ce carnet démontre — un ajout, pas un portage.
2. **Le coût dépasse le budget.** Ces trois cellules portent **la part prépondérante du temps total** du carnet source ; la cible tient en une dizaine de secondes de calcul, et une exécution `evcxr` paie en plus la compilation de ses dépendances. Porter la section ferait d'un carnet de dix secondes un carnet d'une minute pour trois figures.

La suite est donc un **grain déclaré**, avec sa propre page de spécification, et non un contournement silencieux : aucune projection approchée, aucun « t-SNE simplifié » ne prend la place de la section absente.

### Ce que la source établit, et que ce carnet ne rejoue pas

| Aspect | Ce que montre le carnet Python |
|---|---|
| Séparation | La carte t-SNE fait apparaître ~10 îlots quasi purs, là où l'ACP les chevauche : la démonstration visuelle de ce que « non linéaire » apporte |
| Distances | Les distances **entre** îlots et leurs tailles relatives ne sont **pas** interprétables : t-SNE préserve les voisinages locaux, pas la géométrie globale |
| Perplexité | Trois valeurs (5, 30, 50) sur les mêmes données donnent trois cartes ; aucune n'est fausse, la perplexité est un choix d'échelle d'analyse |
| UMAP | Sépare les dix groupes **et** préserve leur arrangement relatif — la différence pratique avec t-SNE — au prix d'un surcoût de compilation sur un jeu de cette taille |

### Le guide de choix, qui reste vrai sans les figures

| Objectif | Méthode | Pourquoi | Limitations à connaître |
|---|---|---|---|
| **Compresser / débruiter** (section 6) | **ACP** | Projection optimale en variance ; la recomposition `mu + z · V` reconstruit une approximation | Linéaire — perd les variations non linéaires |
| **Visualiser / explorer** | **t-SNE / UMAP** | Séparent des classes que l'ACP chevauche | Distances inter-clusters non interprétables (t-SNE) ; perplexité / `n_neighbors` change la carte ; coût super-linéaire (t-SNE) |
| **Réduire AVANT un modèle** | **ACP** | `transform` s'applique à de **nouvelles** données — indispensable en production | t-SNE n'a pas de `transform` : il faut refaire l'embedding entier à chaque nouveau point |

La règle pratique tient en une ligne : **l'ACP est une transformation, t-SNE et UMAP sont des visualisations.** On met la première dans un pipeline, les secondes dans un carnet d'exploration. Pour la lecture des espaces latents (autoencodeurs, VAE, embeddings RAG), ce contraste sera le réflexe à réutiliser.

> **Références de la section.** van der Maaten, L. & Hinton, G. (2008), *Visualizing Data using t-SNE*, Journal of Machine Learning Research 9:2579-2605 ; McInnes, L., Healy, J. & Melville, J. (2018), *UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction*, arXiv:1802.03426.

## 8. Au-delà de KMeans : quand un cluster n'est pas une « boule »

Les sections 2-3 ont clusterisé des **chiffres** avec KMeans : il y a fonctionné, car chaque chiffre forme une masse à peu près compacte. KMeans découpe l'espace avec des **frontières de Voronoï** (chaque point rejoint le centroïde le plus proche) : il produit donc toujours des clusters **convexes**, en « boules ». Dès qu'un groupe authentique ne ressemble pas à une boule — deux croissants imbriqués, deux anneaux concentriques — KMeans le coupe en deux morceaux pourtant arbitraires.

Cette section construit deux jeux de données synthétiques où la **forme** compte, regarde ce que KMeans en fait, puis introduit deux familles qui suivent la forme : DBSCAN (densité) et le **clustering hiérarchique**.

> **Note.** Les générateurs sont ceux du jumeau [2.7](2.7-Modeles-Non-Parametriques-Rust.ipynb) : deux demi-cercles et deux cercles concentriques tracés à la main, bruit gaussien d'un xorshift64 de graine 42. `smartcore::dataset::generator` fournit bien `make_moons_with_seed` et `make_circles_with_seed`, mais son générateur et sa disposition diffèrent de ceux de scikit-learn ; la série a donc tranché pour **un seul** générateur maison, partagé par ses deux carnets Rust. Conséquence à garder en tête : les nuages ne sont pas ceux de la source. Ce qui se compare d'un carnet à l'autre est le **régime** (KMeans échoue, DBSCAN réussit), pas le tirage — les tolérances sur l'ARI sont de 0,20 pour cette raison.

L'ARI (*Adjusted Rand Index*) mesure l'accord entre une partition et les vraies classes : 1 = accord parfait, 0 = accord dû au hasard. `smartcore` n'en fournit pas ; il est écrit dans la cellule (formule de Hubert-Arabie) et vérifié par deux témoins — partition identique → 1, renommage des étiquettes → 1.

In [12]:
use smartcore::api::{Transformer, UnsupervisedEstimator};
use smartcore::cluster::kmeans::{KMeans, KMeansParameters};
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::preprocessing::numerical::{StandardScaler, StandardScalerParameters};

// Generateur de la serie, le meme que le jumeau 2.7 : xorshift64, graine explicite.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

fn make_circles(n: usize, ecart: f64, facteur: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_ext as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_int as f64;
        x.push(vec![facteur * t.cos(), facteur * t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// DBSCAN et le hierarchique mesurent des DISTANCES : l'echelle des axes change tout.
fn standardiser(x: &DenseMatrix<f64>) -> DenseMatrix<f64> {
    let scaler = StandardScaler::<f64>::fit(x, StandardScalerParameters::default()).unwrap();
    scaler.transform(x).unwrap()
}

// ARI (indice de Rand ajuste, formule de Hubert-Arabie) : smartcore n'en fournit pas.
fn ari(a: &[u32], b: &[u32]) -> f64 {
    let mut classes = a.to_vec();
    classes.sort_unstable();
    classes.dedup();
    let mut clusters = b.to_vec();
    clusters.sort_unstable();
    clusters.dedup();
    let (r, c) = (classes.len(), clusters.len());
    let mut nij = vec![0.0_f64; r * c];
    let mut ai = vec![0.0_f64; r];
    let mut bj = vec![0.0_f64; c];
    for i in 0..a.len() {
        let p = classes.iter().position(|&z| z == a[i]).unwrap();
        let q = clusters.iter().position(|&z| z == b[i]).unwrap();
        nij[p * c + q] += 1.0;
        ai[p] += 1.0;
        bj[q] += 1.0;
    }
    let paires = |v: f64| v * (v - 1.0) / 2.0;
    let ij: f64 = nij.iter().map(|&v| paires(v)).sum();
    let si: f64 = ai.iter().map(|&v| paires(v)).sum();
    let sj: f64 = bj.iter().map(|&v| paires(v)).sum();
    let attendu = si * sj / paires(a.len() as f64);
    let maximum = 0.5 * (si + sj);
    if (maximum - attendu).abs() < 1e-12 {
        1.0
    } else {
        (ij - attendu) / (maximum - attendu)
    }
}

// Deux references numeriques pour l'ARI maison : partition identique -> 1,
// renommage des etiquettes -> 1 (l'indice est invariant par permutation).
fn temoins_ari(y: &[u32]) {
    let renomme: Vec<u32> = y.iter().map(|&v| 1 - v).collect();
    println!("Temoin ARI(verite, verite)    : {:.2}", ari(y, y));
    println!("Temoin ARI(verite, renommage) : {:.2}", ari(y, &renomme));
}

// Graine explicite : k-means++ tire son initialisation d'un generateur seme.
fn kmeans_lab(x: &DenseMatrix<f64>, k: usize) -> Vec<u32> {
    let mut params = KMeansParameters::default().with_k(k).with_max_iter(100);
    params.seed = Some(42);
    KMeans::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(x, params)
        .unwrap()
        .predict(x)
        .unwrap()
}

let (x_moons, y_moons) = make_moons(300, 0.10, 42);
let (x_rings, y_rings) = make_circles(300, 0.03, 0.5, 42);
let x_moons = standardiser(&x_moons);
let x_rings = standardiser(&x_rings);

// KMeans force k=2 : il ne peut que decouper en deux "boules" a frontiere droite.
let lab_moons = kmeans_lab(&x_moons, 2);
let lab_rings = kmeans_lab(&x_rings, 2);

println!("ARI KMeans - demi-lunes : {:.2}", ari(&y_moons, &lab_moons));
println!("ARI KMeans - anneaux   : {:.2}", ari(&y_rings, &lab_rings));

temoins_ari(&y_moons);

ARI KMeans - demi-lunes : 0.51


ARI KMeans - anneaux   : -0.00


Temoin ARI(verite, verite)    : 1.00


Temoin ARI(verite, renommage) : 1.00


### Lecture du résultat — pourquoi KMeans échoue ici

Un ARI proche de 0,5 sur les demi-lunes et proche de 0 sur les anneaux veut dire que KMeans ne **reproduit pas** les deux groupes réels. Sur les demi-lunes, il apparie la moitié de chaque croissant : le haut-gauche de l'un finit dans le même cluster que le haut-gauche de l'autre. Sur les anneaux, il assigne un « coin » du cercle extérieur à un cluster et le reste à l'autre. C'est un **échec structurel** : la frontière de Voronoï est toujours droite, donc le cluster est toujours convexe, et ces deux jeux ne le sont pas.

Ce n'est pas une mauvaise exécution, c'est l'**hypothèse** du modèle. Pour suivre une forme, il faut un algorithme qui raisonne en **distance locale** (qui est près de qui) plutôt qu'en **distance au centroïde** (qui est près du centre).

Les deux témoins affichés en fin de cellule ne mesurent pas le clustering : ils mesurent l'**instrument**. Un ARI de 1 sur une partition comparée à elle-même et sur la même partition renommée est le minimum qu'on exige d'une implémentation ; sans ces deux lignes, un ARI de 0,48 ne prouverait rien — il pourrait être faux. C'est la raison pour laquelle l'ARI est écrit dans le carnet plutôt qu'emprunté à l'aveugle.

## 9. DBSCAN : le clustering par densité

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) ne fixe **pas** de nombre de clusters : il part d'un point « noyau » (au moins `min_samples` voisins dans un rayon `eps`) et **étend** le cluster aux points atteignables. Deux conséquences :

- il détecte des formes **arbitraires**, pas seulement des boules ;
- il ne force **aucun** `k` — mais il introduit un hyperparamètre `eps` (le rayon) à régler, et il peut laisser des points **non assignés** (bruit), traités comme des outliers.

> **Note.** `DBSCAN` de `smartcore` ne rend pas les étiquettes de son entraînement : `predict` les recalcule, **en base 1**, avec `0` pour le bruit (`1..k` pour les clusters). scikit-learn rend `-1` pour le bruit et `0..k-1` pour les clusters. La cellule ramène le bruit à `u32::MAX`, qui joue exactement le rôle du `-1` de la source : une étiquette à part entière, que l'ARI traite comme un groupe de plus — ce que fait scikit-learn aussi.

In [13]:
use smartcore::api::{Transformer, UnsupervisedEstimator};
use smartcore::cluster::dbscan::{DBSCAN, DBSCANParameters};
use smartcore::cluster::kmeans::{KMeans, KMeansParameters};
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::preprocessing::numerical::{StandardScaler, StandardScalerParameters};

// Generateur de la serie, le meme que le jumeau 2.7 : xorshift64, graine explicite.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

fn make_circles(n: usize, ecart: f64, facteur: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_ext as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_int as f64;
        x.push(vec![facteur * t.cos(), facteur * t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// DBSCAN et le hierarchique mesurent des DISTANCES : l'echelle des axes change tout.
fn standardiser(x: &DenseMatrix<f64>) -> DenseMatrix<f64> {
    let scaler = StandardScaler::<f64>::fit(x, StandardScalerParameters::default()).unwrap();
    scaler.transform(x).unwrap()
}

// ARI (indice de Rand ajuste, formule de Hubert-Arabie) : smartcore n'en fournit pas.
fn ari(a: &[u32], b: &[u32]) -> f64 {
    let mut classes = a.to_vec();
    classes.sort_unstable();
    classes.dedup();
    let mut clusters = b.to_vec();
    clusters.sort_unstable();
    clusters.dedup();
    let (r, c) = (classes.len(), clusters.len());
    let mut nij = vec![0.0_f64; r * c];
    let mut ai = vec![0.0_f64; r];
    let mut bj = vec![0.0_f64; c];
    for i in 0..a.len() {
        let p = classes.iter().position(|&z| z == a[i]).unwrap();
        let q = clusters.iter().position(|&z| z == b[i]).unwrap();
        nij[p * c + q] += 1.0;
        ai[p] += 1.0;
        bj[q] += 1.0;
    }
    let paires = |v: f64| v * (v - 1.0) / 2.0;
    let ij: f64 = nij.iter().map(|&v| paires(v)).sum();
    let si: f64 = ai.iter().map(|&v| paires(v)).sum();
    let sj: f64 = bj.iter().map(|&v| paires(v)).sum();
    let attendu = si * sj / paires(a.len() as f64);
    let maximum = 0.5 * (si + sj);
    if (maximum - attendu).abs() < 1e-12 {
        1.0
    } else {
        (ij - attendu) / (maximum - attendu)
    }
}

// Deux references numeriques pour l'ARI maison : partition identique -> 1,
// renommage des etiquettes -> 1 (l'indice est invariant par permutation).
fn temoins_ari(y: &[u32]) {
    let renomme: Vec<u32> = y.iter().map(|&v| 1 - v).collect();
    println!("Temoin ARI(verite, verite)    : {:.2}", ari(y, y));
    println!("Temoin ARI(verite, renommage) : {:.2}", ari(y, &renomme));
}

// Graine explicite : k-means++ tire son initialisation d'un generateur seme.
fn kmeans_lab(x: &DenseMatrix<f64>, k: usize) -> Vec<u32> {
    let mut params = KMeansParameters::default().with_k(k).with_max_iter(100);
    params.seed = Some(42);
    KMeans::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(x, params)
        .unwrap()
        .predict(x)
        .unwrap()
}

// Le bruit (0 chez smartcore) devient u32::MAX : l'etiquette a part entiere du -1 de sklearn.
fn dbscan_etiquettes(x: &DenseMatrix<f64>, eps: f64, min_samples: usize) -> Vec<u32> {
    let labels: Vec<u32> = DBSCAN::fit(
        x,
        DBSCANParameters::default()
            .with_eps(eps)
            .with_min_samples(min_samples),
    )
    .and_then(|db| db.predict(x))
    .unwrap();
    labels
        .iter()
        .map(|&v| if v == 0 { u32::MAX } else { v })
        .collect()
}

// Nombre de clusters (hors bruit) et nombre d'outliers.
fn compter(labels: &[u32]) -> (usize, usize) {
    let mut v: Vec<u32> = labels.iter().cloned().filter(|&x| x != u32::MAX).collect();
    v.sort_unstable();
    v.dedup();
    let outliers = labels.iter().filter(|&&x| x == u32::MAX).count();
    (v.len(), outliers)
}

let (x_moons, y_moons) = make_moons(300, 0.10, 42);
let (x_rings, y_rings) = make_circles(300, 0.03, 0.5, 42);
let x_moons = standardiser(&x_moons);
let x_rings = standardiser(&x_rings);

for eps in [0.2, 0.3, 0.4] {
    let lab = dbscan_etiquettes(&x_moons, eps, 5);
    let (ncl, out) = compter(&lab);
    println!(
        "  Demi-lunes eps={:.1} : ARI={:.2}  clusters={}  outliers={}",
        eps,
        ari(&y_moons, &lab),
        ncl,
        out
    );
}

let db_moons = dbscan_etiquettes(&x_moons, 0.3, 5);
let db_rings = dbscan_etiquettes(&x_rings, 0.3, 5);
println!(
    "ARI DBSCAN - demi-lunes : {:.2}  (outliers={})",
    ari(&y_moons, &db_moons),
    compter(&db_moons).1
);
println!(
    "ARI DBSCAN - anneaux   : {:.2}  (outliers={})",
    ari(&y_rings, &db_rings),
    compter(&db_rings).1
);

  Demi-lunes eps=0.2 : ARI=0.58  clusters=7  outliers=15


  Demi-lunes eps=0.3 : ARI=0.99  clusters=2  outliers=1


  Demi-lunes eps=0.4 : ARI=0.00  clusters=1  outliers=0


ARI DBSCAN - demi-lunes : 0.99  (outliers=1)


ARI DBSCAN - anneaux   : 1.00  (outliers=0)


### Lecture du résultat — densité, `eps` et outliers

À `eps=0.3`, DBSCAN retrouve **les deux** formes (ARI proche de 1) : chaque croissant, chaque anneau devient un cluster qui **épouse la forme**. Le balayage `eps` montre le mécanisme : trop petit, le cluster se fragmente et une partie des points devient du bruit (outliers) ; trop grand, tout se fond en **un seul** cluster (ARI proche de 0). On n'a **pas** à choisir `k`, mais on a **remplacé `k` par `eps`** — le réglage reste, c'est juste un paramètre différent, plus attaché à l'échelle des données (d'où la standardisation du début de section).

Le bruit est un avantage d'honnêteté : là où KMeans assigne *tout* le monde, DBSCAN avoue qu'un point est du bruit s'il n'est dense près de rien.

> **Ce que le chiffre ne dit pas.** Un `eps` plus grand rassemble plus de points, donc son ARI peut monter sur une forme puis s'effondrer sur l'autre : `eps` est **une** valeur pour **un** jeu de données, et les trois lignes du balayage sont trois réglages différents, pas trois mesures de la qualité de DBSCAN.

### Exercice 5 : trouver le rayon `eps` qui fait la balance

La réussite de DBSCAN dépend du rayon `eps`. Fais-le varier sur les demi-lunes et observe l'ARI, le nombre de clusters et le nombre d'outliers. Trouve le `eps` qui sépare les deux croissants **sans** les fragmenter.

La cellule balaie déjà sept valeurs et affiche les trois grandeurs ; l'exercice est de **décider** quel critère définit « sépare sans fragmenter », puis d'écrire la réponse sous forme de valeur numérique : le plus petit `eps` dont l'ARI dépasse 0,95 avec exactement deux clusters.

**Indice** : une fragmentation se lit au nombre de clusters (plus de deux), un effondrement à l'ARI (proche de 0 avec un seul cluster). Les valeurs intermédiaires où les deux défauts coexistent ne conviennent pas.

In [14]:
use smartcore::api::{Transformer, UnsupervisedEstimator};
use smartcore::cluster::dbscan::{DBSCAN, DBSCANParameters};
use smartcore::cluster::kmeans::{KMeans, KMeansParameters};
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::preprocessing::numerical::{StandardScaler, StandardScalerParameters};

// Generateur de la serie, le meme que le jumeau 2.7 : xorshift64, graine explicite.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

fn make_circles(n: usize, ecart: f64, facteur: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_ext as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_int as f64;
        x.push(vec![facteur * t.cos(), facteur * t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// DBSCAN et le hierarchique mesurent des DISTANCES : l'echelle des axes change tout.
fn standardiser(x: &DenseMatrix<f64>) -> DenseMatrix<f64> {
    let scaler = StandardScaler::<f64>::fit(x, StandardScalerParameters::default()).unwrap();
    scaler.transform(x).unwrap()
}

// ARI (indice de Rand ajuste, formule de Hubert-Arabie) : smartcore n'en fournit pas.
fn ari(a: &[u32], b: &[u32]) -> f64 {
    let mut classes = a.to_vec();
    classes.sort_unstable();
    classes.dedup();
    let mut clusters = b.to_vec();
    clusters.sort_unstable();
    clusters.dedup();
    let (r, c) = (classes.len(), clusters.len());
    let mut nij = vec![0.0_f64; r * c];
    let mut ai = vec![0.0_f64; r];
    let mut bj = vec![0.0_f64; c];
    for i in 0..a.len() {
        let p = classes.iter().position(|&z| z == a[i]).unwrap();
        let q = clusters.iter().position(|&z| z == b[i]).unwrap();
        nij[p * c + q] += 1.0;
        ai[p] += 1.0;
        bj[q] += 1.0;
    }
    let paires = |v: f64| v * (v - 1.0) / 2.0;
    let ij: f64 = nij.iter().map(|&v| paires(v)).sum();
    let si: f64 = ai.iter().map(|&v| paires(v)).sum();
    let sj: f64 = bj.iter().map(|&v| paires(v)).sum();
    let attendu = si * sj / paires(a.len() as f64);
    let maximum = 0.5 * (si + sj);
    if (maximum - attendu).abs() < 1e-12 {
        1.0
    } else {
        (ij - attendu) / (maximum - attendu)
    }
}

// Deux references numeriques pour l'ARI maison : partition identique -> 1,
// renommage des etiquettes -> 1 (l'indice est invariant par permutation).
fn temoins_ari(y: &[u32]) {
    let renomme: Vec<u32> = y.iter().map(|&v| 1 - v).collect();
    println!("Temoin ARI(verite, verite)    : {:.2}", ari(y, y));
    println!("Temoin ARI(verite, renommage) : {:.2}", ari(y, &renomme));
}

// Graine explicite : k-means++ tire son initialisation d'un generateur seme.
fn kmeans_lab(x: &DenseMatrix<f64>, k: usize) -> Vec<u32> {
    let mut params = KMeansParameters::default().with_k(k).with_max_iter(100);
    params.seed = Some(42);
    KMeans::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(x, params)
        .unwrap()
        .predict(x)
        .unwrap()
}

fn dbscan_etiquettes(x: &DenseMatrix<f64>, eps: f64, min_samples: usize) -> Vec<u32> {
    let labels: Vec<u32> = DBSCAN::fit(
        x,
        DBSCANParameters::default()
            .with_eps(eps)
            .with_min_samples(min_samples),
    )
    .and_then(|db| db.predict(x))
    .unwrap();
    labels
        .iter()
        .map(|&v| if v == 0 { u32::MAX } else { v })
        .collect()
}

fn compter(labels: &[u32]) -> (usize, usize) {
    let mut v: Vec<u32> = labels.iter().cloned().filter(|&x| x != u32::MAX).collect();
    v.sort_unstable();
    v.dedup();
    let outliers = labels.iter().filter(|&&x| x == u32::MAX).count();
    (v.len(), outliers)
}

let (x_moons, y_moons) = make_moons(300, 0.10, 42);
let x_moons = standardiser(&x_moons);

println!("{:>5}  {:>6}  {:>8}  {:>8}", "eps", "ARI", "clusters", "outliers");
for eps in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45] {
    let lab = dbscan_etiquettes(&x_moons, eps, 5);
    let (ncl, out) = compter(&lab);
    println!(
        "{:>5.2}  {:>6.2}  {:>8}  {:>8}",
        eps,
        ari(&y_moons, &lab),
        ncl,
        out
    );
}

// TODO etudiant : plus petit eps dont l'ARI depasse 0.95 avec exactement 2 clusters
let bon_eps: Option<f64> = None;

match bon_eps {
    Some(v) => println!("Exercice 5 : eps = {:.2} separe les deux croissants sans les fragmenter.", v),
    None => println!(
        "Exercice 5 a completer : bon_eps vaut None (indice : lire la table, garder 2 clusters et l'ARI le plus haut)."
    ),
}

  eps     ARI  clusters  outliers


 0.15    0.08        18        80


 0.20    0.58         7        15


 0.25    0.82         4         4


 0.30    0.99         2         1


 0.35    0.99         2         1


 0.40    0.00         1         0


 0.45    0.00         1         0


Exercice 5 a completer : bon_eps vaut None (indice : lire la table, garder 2 clusters et l'ARI le plus haut).


()

## 10. Le clustering hiérarchique

Le clustering hiérarchique **agglomératif** construit une hiérarchie : on part de chaque point isolé et on **fusionne** les deux groupes les plus proches, jusqu'à un seul. Le résultat est un **dendrogramme** (arbre de fusion), que l'on découpe à la hauteur donnant le bon nombre de clusters. Le choix du **lien** (distance entre deux *groupes*) change tout :

- **lien simple** (single) : la distance la plus courte entre les deux groupes — suit les formes allongées, mais « enchaîne » des points via un pont ;
- **lien complet** (complete) : la distance la plus longue — produit des groupes compacts, mais rate les formes allongées.

> **Deux écarts déclarés.** `smartcore` 0.7.0 n'implémente **que le lien simple** : la comparaison simple / complet de la source ne peut pas être rejouée, et ce carnet n'en fabrique pas un substitut. Et il n'expose **aucune hauteur de fusion** : le dendrogramme ne peut pas être reconstruit, pas même en table — les distances successives ne sont pas lisibles depuis l'objet. À la place, la coupe est pilotée par `n_clusters`, et l'exercice 6 fait varier cette coupe pour montrer où la structure se sépare. Le lien complet est donc **absent**, pas approximé : c'est la même position que pour la section 7.

In [15]:
use smartcore::api::{Transformer, UnsupervisedEstimator};
use smartcore::cluster::agglomerative::{
    AgglomerativeClustering, AgglomerativeClusteringParameters,
};
use smartcore::cluster::kmeans::{KMeans, KMeansParameters};
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::preprocessing::numerical::{StandardScaler, StandardScalerParameters};

// Generateur de la serie, le meme que le jumeau 2.7 : xorshift64, graine explicite.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

fn make_circles(n: usize, ecart: f64, facteur: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_ext as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_int as f64;
        x.push(vec![facteur * t.cos(), facteur * t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// DBSCAN et le hierarchique mesurent des DISTANCES : l'echelle des axes change tout.
fn standardiser(x: &DenseMatrix<f64>) -> DenseMatrix<f64> {
    let scaler = StandardScaler::<f64>::fit(x, StandardScalerParameters::default()).unwrap();
    scaler.transform(x).unwrap()
}

// ARI (indice de Rand ajuste, formule de Hubert-Arabie) : smartcore n'en fournit pas.
fn ari(a: &[u32], b: &[u32]) -> f64 {
    let mut classes = a.to_vec();
    classes.sort_unstable();
    classes.dedup();
    let mut clusters = b.to_vec();
    clusters.sort_unstable();
    clusters.dedup();
    let (r, c) = (classes.len(), clusters.len());
    let mut nij = vec![0.0_f64; r * c];
    let mut ai = vec![0.0_f64; r];
    let mut bj = vec![0.0_f64; c];
    for i in 0..a.len() {
        let p = classes.iter().position(|&z| z == a[i]).unwrap();
        let q = clusters.iter().position(|&z| z == b[i]).unwrap();
        nij[p * c + q] += 1.0;
        ai[p] += 1.0;
        bj[q] += 1.0;
    }
    let paires = |v: f64| v * (v - 1.0) / 2.0;
    let ij: f64 = nij.iter().map(|&v| paires(v)).sum();
    let si: f64 = ai.iter().map(|&v| paires(v)).sum();
    let sj: f64 = bj.iter().map(|&v| paires(v)).sum();
    let attendu = si * sj / paires(a.len() as f64);
    let maximum = 0.5 * (si + sj);
    if (maximum - attendu).abs() < 1e-12 {
        1.0
    } else {
        (ij - attendu) / (maximum - attendu)
    }
}

// Deux references numeriques pour l'ARI maison : partition identique -> 1,
// renommage des etiquettes -> 1 (l'indice est invariant par permutation).
fn temoins_ari(y: &[u32]) {
    let renomme: Vec<u32> = y.iter().map(|&v| 1 - v).collect();
    println!("Temoin ARI(verite, verite)    : {:.2}", ari(y, y));
    println!("Temoin ARI(verite, renommage) : {:.2}", ari(y, &renomme));
}

// Graine explicite : k-means++ tire son initialisation d'un generateur seme.
fn kmeans_lab(x: &DenseMatrix<f64>, k: usize) -> Vec<u32> {
    let mut params = KMeansParameters::default().with_k(k).with_max_iter(100);
    params.seed = Some(42);
    KMeans::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(x, params)
        .unwrap()
        .predict(x)
        .unwrap()
}

// smartcore 0.7.0 n'implemente que le lien simple : il n'y a pas de parametre de lien.
// `labels` est public sur l'objet ajuste (Vec<usize>).
fn hier_lab(x: &DenseMatrix<f64>, n_clusters: usize) -> Vec<u32> {
    let ag = AgglomerativeClustering::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(
        x,
        AgglomerativeClusteringParameters::default().with_n_clusters(n_clusters),
    )
    .unwrap();
    ag.labels.iter().map(|&v| v as u32).collect()
}

let (x_moons, y_moons) = make_moons(300, 0.10, 42);
let (x_rings, y_rings) = make_circles(300, 0.03, 0.5, 42);
let x_moons = standardiser(&x_moons);
let x_rings = standardiser(&x_rings);

let ag_moons = hier_lab(&x_moons, 2);
let ag_rings = hier_lab(&x_rings, 2);

println!(
    "ARI hierarchique lien simple - demi-lunes : {:.2}",
    ari(&y_moons, &ag_moons)
);
println!(
    "ARI hierarchique lien simple - anneaux   : {:.2}",
    ari(&y_rings, &ag_rings)
);

ARI hierarchique lien simple - demi-lunes : 1.00


ARI hierarchique lien simple - anneaux   : 1.00


### Lecture du résultat — chaining et coupe

Le **lien simple** fusionne d'abord les points les plus proches : sur les anneaux, il commence par coller les points voisins le long du cercle, puis, quand deux points presque voisins appartiennent à deux anneaux différents, la fusion saute le vide entre les deux cercles. Les deux anneaux finissent donc chacun en une branche, et couper l'arbre à deux groupes les sépare exactement (ARI = 1).

C'est le phénomène de **chaining** : le lien simple suit les formes courbes parce qu'il n'exige qu'*un* voisin proche, mais un pont de quelques points suffit à souder deux groupes qui ne devraient pas l'être. Le lien complet, lui, exige que deux groupes soient proches *en tous points* (distance la plus longue) ; sur un anneau creux, les points les plus éloignés sont à l'opposé l'un de l'autre, et il coupe mal. La source le mesure, ce carnet ne peut pas le rejouer — le lien complet n'existant pas dans la bibliothèque.

Il n'y a pas de choix gratuit : c'est le sens du guide qui suit.

### Exercice 6 : où couper la hiérarchie ?

**Objectif** : lire l'effet du **seuil de coupe** sur les anneaux, seul levier disponible puisque la bibliothèque n'expose pas le choix du lien.

La cellule balaie `n_clusters` de 2 à 5 sur les deux formes et affiche l'ARI. Deux questions, deux réponses à écrire :

1. `meilleur_n` : le nombre de groupes que la coupe doit demander pour retrouver la structure des **anneaux** ;
2. `forme_limite` : laquelle des deux formes serait trahie par un pont de quelques points entre les groupes — `"demi-lunes"` ou `"anneaux"` ?

**Indice** : sur les anneaux, demander plus de deux groupes découpe un anneau en arcs, donc l'ARI baisse. Un pont de bruit entre deux croissants soude les demi-lunes en une seule chaîne sous le lien simple, ce que la coupe ne peut plus défaire.

**Note.** La source comparait deux liens ; l'exercice porte ici sur la **coupe**, parce que c'est le seul levier que la bibliothèque offre. L'intention pédagogique — « le lien et la coupe changent le résultat » — est conservée, la grandeur demandée change.

In [16]:
use smartcore::api::{Transformer, UnsupervisedEstimator};
use smartcore::cluster::agglomerative::{
    AgglomerativeClustering, AgglomerativeClusteringParameters,
};
use smartcore::cluster::kmeans::{KMeans, KMeansParameters};
use smartcore::linalg::basic::arrays::Array;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::preprocessing::numerical::{StandardScaler, StandardScalerParameters};

// Generateur de la serie, le meme que le jumeau 2.7 : xorshift64, graine explicite.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

fn make_circles(n: usize, ecart: f64, facteur: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_ext as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_int as f64;
        x.push(vec![facteur * t.cos(), facteur * t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// DBSCAN et le hierarchique mesurent des DISTANCES : l'echelle des axes change tout.
fn standardiser(x: &DenseMatrix<f64>) -> DenseMatrix<f64> {
    let scaler = StandardScaler::<f64>::fit(x, StandardScalerParameters::default()).unwrap();
    scaler.transform(x).unwrap()
}

// ARI (indice de Rand ajuste, formule de Hubert-Arabie) : smartcore n'en fournit pas.
fn ari(a: &[u32], b: &[u32]) -> f64 {
    let mut classes = a.to_vec();
    classes.sort_unstable();
    classes.dedup();
    let mut clusters = b.to_vec();
    clusters.sort_unstable();
    clusters.dedup();
    let (r, c) = (classes.len(), clusters.len());
    let mut nij = vec![0.0_f64; r * c];
    let mut ai = vec![0.0_f64; r];
    let mut bj = vec![0.0_f64; c];
    for i in 0..a.len() {
        let p = classes.iter().position(|&z| z == a[i]).unwrap();
        let q = clusters.iter().position(|&z| z == b[i]).unwrap();
        nij[p * c + q] += 1.0;
        ai[p] += 1.0;
        bj[q] += 1.0;
    }
    let paires = |v: f64| v * (v - 1.0) / 2.0;
    let ij: f64 = nij.iter().map(|&v| paires(v)).sum();
    let si: f64 = ai.iter().map(|&v| paires(v)).sum();
    let sj: f64 = bj.iter().map(|&v| paires(v)).sum();
    let attendu = si * sj / paires(a.len() as f64);
    let maximum = 0.5 * (si + sj);
    if (maximum - attendu).abs() < 1e-12 {
        1.0
    } else {
        (ij - attendu) / (maximum - attendu)
    }
}

// Deux references numeriques pour l'ARI maison : partition identique -> 1,
// renommage des etiquettes -> 1 (l'indice est invariant par permutation).
fn temoins_ari(y: &[u32]) {
    let renomme: Vec<u32> = y.iter().map(|&v| 1 - v).collect();
    println!("Temoin ARI(verite, verite)    : {:.2}", ari(y, y));
    println!("Temoin ARI(verite, renommage) : {:.2}", ari(y, &renomme));
}

// Graine explicite : k-means++ tire son initialisation d'un generateur seme.
fn kmeans_lab(x: &DenseMatrix<f64>, k: usize) -> Vec<u32> {
    let mut params = KMeansParameters::default().with_k(k).with_max_iter(100);
    params.seed = Some(42);
    KMeans::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(x, params)
        .unwrap()
        .predict(x)
        .unwrap()
}

fn hier_lab(x: &DenseMatrix<f64>, n_clusters: usize) -> Vec<u32> {
    let ag = AgglomerativeClustering::<f64, u32, DenseMatrix<f64>, Vec<u32>>::fit(
        x,
        AgglomerativeClusteringParameters::default().with_n_clusters(n_clusters),
    )
    .unwrap();
    ag.labels.iter().map(|&v| v as u32).collect()
}

let (x_moons, y_moons) = make_moons(300, 0.10, 42);
let (x_rings, y_rings) = make_circles(300, 0.03, 0.5, 42);
let x_moons = standardiser(&x_moons);
let x_rings = standardiser(&x_rings);

println!("{:>9}  {:>12}  {:>12}", "n_clusters", "ARI anneaux", "ARI demi-lunes");
for n_clusters in 2..=5 {
    let ann = hier_lab(&x_rings, n_clusters);
    let lun = hier_lab(&x_moons, n_clusters);
    println!(
        "{:>9}  {:>12.2}  {:>12.2}",
        n_clusters,
        ari(&y_rings, &ann),
        ari(&y_moons, &lun)
    );
}

// TODO etudiant : le nombre de groupes que la coupe doit demander sur les anneaux
let meilleur_n: Option<usize> = None;
// TODO etudiant : "demi-lunes" ou "anneaux" -- la forme qu'un pont de points trahirait
let forme_limite: Option<&str> = None;

match (meilleur_n, forme_limite) {
    (Some(n), Some(f)) => println!(
        "Exercice 6 : couper a {} groupes retrouve les anneaux ; un pont de points trahirait les {}.",
        n, f
    ),
    _ => println!(
        "Exercice 6 a completer : meilleur_n = {:?}, forme_limite = {:?} \
         (indice : lire la table, la coupe ou l'ARI des anneaux est maximal).",
        meilleur_n, forme_limite
    ),
}

n_clusters   ARI anneaux  ARI demi-lunes


        2          1.00          1.00


        3          0.75          0.99


        4          0.71          0.99


        5          0.71          0.92


Exercice 6 a completer : meilleur_n = None, forme_limite = None (indice : lire la table, la coupe ou l'ARI des anneaux est maximal).


()

## 11. Guide de choix : quel algorithme pour quelle forme ?

| Algorithme | Hypothèse implicite | Formes qu'il suit | À connaître | Résultat ici (demi-lunes / anneaux) |
|---|---|---|---|---|
| **KMeans** | clusters convexes, isotropes, tailles comparables | « boules » | il faut fixer `k` ; frontières droites ; assigne tout le monde | échec sur les deux formes |
| **DBSCAN** | densité locale : zones denses séparées par du vide | formes arbitraires | il faut régler `eps` (+ `min_samples`) ; bruit = outlier ; fragilité à `eps` | succès sur les deux formes à `eps` = 0,3 |
| **Hiérarchique** | aucune forme imposée a priori | ce que le **lien** autorise | choix du lien (ici : lien simple seulement) ; coupe à `n_clusters` | succès sur les deux formes au lien simple |

Les deux dernières lignes renvoient aux ARI mesurés dans les sections 9 et 10 de **ce** carnet ; la colonne de la source y portait ses propres chiffres, obtenus sur d'autres tirages.

> **Le réflexe.** Avant tout clustering à base de distance (DBSCAN, hiérarchique, et même KMeans), **standardiser** les variables — sinon le réglage d'`eps` ou la distance dépendent de l'unité de chaque axe. Et quand un groupe ne ressemble visiblement pas à une boule, ne pas « tordre » KMeans : **changer de modèle**.

### Exercice 7 : quel algorithme dans quel cas ?

Associe chaque description à la famille la plus adaptée (KMeans, DBSCAN ou hiérarchique) et justifie en une ligne.

- (a) une base clients : 3 masses homogènes et bien séparées ;
- (b) une carte de tâches spatiales : des îlots denses et du bruit.

**Indice** : la question à se poser n'est pas « quel algorithme est le meilleur » mais « quelle hypothèse du modèle mes données respectent-elles ». Des masses compactes et séparées satisfont l'hypothèse de KMeans ; des îlots denses entourés de vide avec des points isolés sont exactement ce que DBSCAN modélise, bruit compris.

In [17]:
// Exercice 7 : quel algorithme pour quelle forme ?
// (a) une base clients : 3 masses homogenes et bien separees       -> ?
// (b) une carte de taches spatiales : des ilots denses et du bruit -> ?
let choix_a: Option<&str> = None; // TODO etudiant : "KMeans", "DBSCAN" ou "hierarchique"
let choix_b: Option<&str> = None; // TODO etudiant : idem

match (choix_a, choix_b) {
    (Some(a), Some(b)) => println!("Exercice 7 : (a) {} / (b) {}", a, b),
    _ => println!(
        "Exercice 7 a completer : (a) {:?}, (b) {:?}",
        choix_a, choix_b
    ),
}

Exercice 7 a completer : (a) None, (b) None


()

## Synthèse : supervisé vs non supervisé

Le **supervisé** (carnets 2.1 à 2.5) dispose d'étiquettes et apprend à **prédire**. Le **non supervisé** (ce carnet) n'a pas d'étiquettes et **trouve** de la structure (clusters) ou **compresse** (PCA). En pratique, ces approches se combinent souvent : on peut appliquer une PCA avant un clustering, ou avant un modèle supervisé pour accélérer l'apprentissage et réduire le surajustement.

> **Référence.** Hastie, T., Tibshirani, R. & Friedman, J. (2009), *The Elements of Statistical Learning*, Springer (2e éd.), §14.3, 14.5. — Synthèse du clustering et de la réduction de dimension dans le cadre de l'apprentissage statistique.

## Conclusion : le socle ML, vu depuis Rust

Vous avez parcouru le socle ML de la série `02-ML-Cours` : (2.1) le workflow ML, (2.2) la descente de gradient, (2.3) la régression linéaire et logistique, (2.4) les arbres / forêts / ensembles, (2.5) le compromis biais-variance / validation croisée / courbes ROC, et (2.6) le clustering, l'ACP et — dans la source seulement — la réduction non linéaire de visualisation (t-SNE, UMAP).

Ce que ce carnet ajoute à la version Python n'est pas un algorithme : c'est la **mesure**. Trois grandeurs que scikit-learn rend d'un appel (`inertia_`, `explained_variance_ratio_`, `inverse_transform`) sont ici recalculées à la main — moyenne par cluster, variance des colonnes projetées, recomposition `mu + z · V` — et une quatrième (`adjusted_rand_score`) est écrite de zéro, avec ses deux témoins. Lire un résultat sans son mode de calcul est le premier pas vers le lire sans le comprendre ; l'écrire est le second.

La série se poursuit avec [2.7](2.7-Modeles-Non-Parametriques.ipynb) (modèles non paramétriques), dont le [jumeau Rust](2.7-Modeles-Non-Parametriques-Rust.ipynb) existe déjà, puis [2.8](2.8-Theorie-PAC.ipynb) (théorie PAC) et [2.9](2.9-Grokking-Generalisation.ipynb) (grokking).

## References

1. MacQueen, J. (1967). *Some Methods for Classification and Analysis of Multivariate Observations*. Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability 1:281-297. — La méthode des k-moyennes, partitionnement par minimisation de l'inertie intra-cluster.
2. Pearson, K. (1901). *On Lines and Planes of Closest Fit to Systems of Points in Space*. Philosophical Magazine 2(11):559-572. — Article fondateur de l'Analyse en Composantes Principales (axes de variance maximale).
3. Jolliffe, I.T. (2002). *Principal Component Analysis*. Springer (2e éd.). — Référence sur l'ACP : variance expliquée, choix du nombre de composantes.
4. Ester, M., Kriegel, H.-P., Sander, J. & Xu, X. (1996). *A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise*. Proceedings of KDD-96:226-231. — DBSCAN : points noyaux, atteignabilité par densité, gestion du bruit.
5. Hubert, L. & Arabie, P. (1985). *Comparing Partitions*. Journal of Classification 2(1):193-218. — L'indice de Rand ajusté, tel qu'implémenté dans ce carnet.
6. van der Maaten, L. & Hinton, G. (2008). *Visualizing Data using t-SNE*. Journal of Machine Learning Research 9:2579-2605. — t-SNE : embedding non linéaire préservant les voisinages, rôle de la perplexité (section non portée, citée pour la source).
7. McInnes, L., Healy, J. & Melville, J. (2018). *UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction*. arXiv:1802.03426. — UMAP : graphe de voisinage, rapidité et conservation de la structure globale (section non portée).
8. Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*. Springer (2e éd.), §14.3, 14.5. — Clustering et réduction de dimension.
9. Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research 12:2825-2830. — `KMeans`, `PCA`, `DBSCAN`, `AgglomerativeClustering`, `StandardScaler`, `make_moons`, `make_circles`, `load_digits`, `adjusted_rand_score` : les organes que ce carnet remplace.

*Implémentation Rust* : crate `smartcore` 0.7.0 (licences MIT / Apache-2.0) — `cluster::kmeans`, `decomposition::pca`, `cluster::dbscan`, `cluster::agglomerative`, `preprocessing::numerical::StandardScaler`, `dataset::digits`. Les générateurs `make_moons` / `make_circles`, l'ARI et la recomposition PCA sont écrits dans le carnet.

## À retenir

KMeans minimise une inertie et attribue des identifiants arbitraires ; la PCA projette selon la variance et ne préserve pas toute l'information. Les formes non convexes motivent DBSCAN ou un lien hiérarchique adapté.

## Vérifiez votre compréhension

1. Un cluster numéro 3 correspond-il forcément au chiffre 3 ?

<details><summary>Réponse 1</summary>

Non : les numéros sont arbitraires ; il faut comparer les groupes aux étiquettes indépendamment de leur permutation.

</details>

2. Que perd-on en projetant les chiffres sur deux composantes ?

<details><summary>Réponse 2</summary>

La variance non captée par ces composantes ; le graphique est une projection et non toute la géométrie.

</details>

3. Pourquoi KMeans peine-t-il sur des anneaux ?

<details><summary>Réponse 3</summary>

Ses affectations aux centres privilégient des régions convexes et ne suivent pas les anneaux imbriqués.

</details>

4. Que change le rayon `eps` de DBSCAN ?

<details><summary>Réponse 4</summary>

Il change le voisinage de densité : trop petit fragmente les groupes, trop grand peut les fusionner.

</details>

5. Pourquoi l'inertie ne suffit-elle pas à choisir `k` ?

<details><summary>Réponse 5</summary>

Elle décroît toujours quand `k` augmente : son minimum est atteint quand chaque point est son propre cluster. Le choix se fait sur le **gain** marginal (le coude), jamais sur la valeur seule.

</details>

## Pour aller plus loin

[Modèles non paramétriques](2.7-Modeles-Non-Parametriques.ipynb) reprend les distances dans un cadre supervisé ; son [jumeau Rust](2.7-Modeles-Non-Parametriques-Rust.ipynb) réutilise le même générateur de formes et la même crate.